# Sliding-window Transformer with KV caching

## 1. Hugging Face Login

In [4]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [6]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 9. Define The Model

In [8]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class WindowConfig:
    vocab_size: int
    pad_id: int
    d_model: int = 512
    n_layers: int = 12
    n_heads: int = 8
    n_kv_heads: int = 2
    d_ff: int = 2048
    max_seq_len: int = 1024  # Maximum training chunk, not a generation position limit.
    window_size: int = 256
    query_chunk_size: int = 128
    dropout: float = 0.1
    rope_base: float = 10000.0

class RMSNorm(nn.Module):
    def __init__(self, width, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.eps = eps

    def forward(self, x):
        # Compute the mean square in float32 for numerical stability.
        normalized = x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps)
        return normalized.to(x.dtype) * self.weight

class RotaryPositions(nn.Module):
    def __init__(self, head_dim, base):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, x, positions):
        # x: [B, heads, T, head_dim]. Rotate neighboring even/odd features.
        with torch.autocast(device_type=x.device.type, enabled=False):
            angles = positions.float()[:, None] * self.inv_freq.float()[None, :]
            cos = angles.cos()[None, None].to(x.dtype)
            sin = angles.sin()[None, None].to(x.dtype)
        even, odd = x[..., 0::2], x[..., 1::2]
        return torch.stack((even*cos - odd*sin, even*sin + odd*cos), dim=-1).flatten(-2)

class SlidingWindowGQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.head_dim = cfg.d_model // cfg.n_heads
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.group_size = cfg.n_heads // cfg.n_kv_heads
        self.window = cfg.window_size
        self.chunk_size = min(cfg.query_chunk_size, cfg.window_size)
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads*self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads*self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.rope = RotaryPositions(self.head_dim, cfg.rope_base)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, positions, key_valid, past=None, use_cache=False):
        b, t, _ = x.shape
        q = self.q_proj(x).view(b,t,self.n_heads,self.head_dim).transpose(1,2)
        k = self.k_proj(x).view(b,t,self.n_kv_heads,self.head_dim).transpose(1,2)
        v = self.v_proj(x).view(b,t,self.n_kv_heads,self.head_dim).transpose(1,2)
        q, k = self.rope(q,positions), self.rope(k,positions)
        past_length = 0 if past is None else past[0].size(2)
        if past is not None:
            k = torch.cat([past[0],k],dim=2)
            v = torch.cat([past[1],v],dim=2)
        q = q.reshape(b,self.n_kv_heads,self.group_size,t,self.head_dim)
        outputs = []
        # Process blocks of queries against ONLY the nearby keys.
        # A score block has at most chunk_size x (window + chunk_size - 1) entries.
        for start in range(0,t,self.chunk_size):
            end = min(start+self.chunk_size,t)
            left = max(0,past_length+start-self.window+1)
            right = past_length+end
            local_k, local_v = k[:,:,left:right], v[:,:,left:right]
            query_index = torch.arange(past_length+start,past_length+end,device=x.device)
            key_index = torch.arange(left,right,device=x.device)
            visible = (key_index[None,:] <= query_index[:,None]) & (
                key_index[None,:] > query_index[:,None]-self.window)
            allowed = visible[None] & key_valid[:,None,left:right]
            # Right-padded query rows can have no real keys in their window.
            # Give only those ignored rows a fallback key to avoid NaN softmax.
            fallback = torch.zeros_like(allowed)
            fallback[:,:,0] = ~allowed.any(dim=-1)
            allowed = allowed | fallback
            with torch.autocast(device_type=x.device.type,enabled=False):
                scores = torch.einsum('bgrtd,bgsd->bgrts',q[:,:,:,start:end].float(),local_k.float())
                scores = scores / math.sqrt(self.head_dim)
                weights = scores.masked_fill(~allowed[:,None,None],float('-inf')).softmax(-1)
            context = torch.einsum('bgrts,bgsd->bgrtd',self.dropout(weights.to(v.dtype)),local_v)
            outputs.append(context)
        context = torch.cat(outputs,dim=3).reshape(b,self.n_heads,t,self.head_dim)
        context = context.transpose(1,2).contiguous().view(b,t,-1)
        new_cache = None
        if use_cache:
            # Keep w-1 previous tokens; the next token supplies the w-th key.
            # clone releases the larger prefill storage instead of retaining a view.
            keep = self.window-1
            new_cache = (k[:,:,-keep:].clone(),v[:,:,-keep:].clone()) if keep else (k[:,:,:0].clone(),v[:,:,:0].clone())
        return self.out_proj(context), new_cache

In [9]:
class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.norm2 = RMSNorm(cfg.d_model)
        self.attention = SlidingWindowGQA(cfg)
        self.ffn = SwiGLU(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, positions, key_valid, past=None, use_cache=False):
        attended, cache = self.attention(self.norm1(x), positions, key_valid, past, use_cache)
        x = x + self.dropout(attended)
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x, cache

class WindowGQALM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        if (cfg.d_model % cfg.n_heads or cfg.n_heads % cfg.n_kv_heads
                or (cfg.d_model//cfg.n_heads) % 2 or cfg.window_size < 1
                or cfg.query_chunk_size < 1):
            raise ValueError('Check head dimensions, window size and query chunk size')
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self.initialize)
        # Tie AFTER initialization so the shared parameter is initialized once.
        self.lm_head.weight = self.embedding.weight
        with torch.no_grad():
            self.embedding.weight[cfg.pad_id].zero_()
        for block in self.blocks:
            nn.init.normal_(block.attention.out_proj.weight, std=0.02/math.sqrt(2*cfg.n_layers))
            nn.init.normal_(block.ffn.down.weight, std=0.02/math.sqrt(2*cfg.n_layers))

    @staticmethod
    def initialize(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, ids, cache=None, use_cache=False, last_only=False):
        if ids.ndim != 2 or ids.size(1) == 0:
            raise ValueError('Use nonempty [batch, time] token IDs')
        if cache is not None and not use_cache:
            raise ValueError('A cache requires use_cache=True')
        if use_cache and self.training:
            raise ValueError('Call eval() before cached generation')
        if self.training and ids.size(1) > self.cfg.max_seq_len:
            raise ValueError('Training batch exceeds configured sequence length')
        # Absolute RoPE offset is independent of the bounded cache length.
        offset = 0 if cache is None else cache['position']
        positions = torch.arange(offset,offset+ids.size(1),device=ids.device)
        valid = ids.ne(self.cfg.pad_id)
        if cache is not None:
            valid = torch.cat([cache['valid'],valid],dim=1)
        x = self.dropout(self.embedding(ids))
        updated = []
        for index, block in enumerate(self.blocks):
            previous = None if cache is None else cache['kv'][index]
            x, new_kv = block(x,positions,valid,previous,use_cache)
            if use_cache: updated.append(new_kv)
        logits = self.lm_head(self.final_norm(x[:, -1:] if last_only else x))
        if use_cache:
            keep = self.cfg.window_size-1
            return logits, {'kv':updated,
                'valid':valid[:,-keep:].clone() if keep else valid[:,:0].clone(),
                'position':offset+ids.size(1)}
        return logits

# **Phase 3**

In [2]:
import os
import json
import random
import zipfile

from pathlib import Path
from dataclasses import asdict

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import (
    precision_recall_fscore_support,
    classification_report,
)


MODEL_DIR = Path(
    "/kaggle/input/datasets/bokkatrishalreddy/efficient-for-phase3-files"
)

METHOD = "lora"                # "lora" or "lorma_plus"

USE_CONTEXT = True
USE_POSITION = False
USE_CLASS_WEIGHTS = False

MAX_LENGTH = 512
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 1e-4
PATIENCE = 3

RANK = 8
ALPHA = 16
SEED = 42

ATTENTION_TARGETS = [
    "q_proj",
    "k_proj",
    "v_proj",
    "out_proj",
]

FFN_TARGETS = ["gate", "up", "down"]

RUN_NAME = (
    f"efficient_{METHOD}_mean_eos"
    f"_position{int(USE_POSITION)}"
)

OUTPUT_DIR = Path("/kaggle/working/phase3") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

use_amp = device.type == "cuda"

amp_dtype = (
    torch.bfloat16
    if use_amp and torch.cuda.is_bf16_supported()
    else torch.float16
)

print("Device:", device)
print("Method:", METHOD)
print("Output:", OUTPUT_DIR)

Device: cuda
Method: lora
Output: /kaggle/working/phase3/efficient_lora_mean_eos_position0


In [10]:
from huggingface_hub import hf_hub_download


DATA_DIR = Path("/kaggle/working/capstone_data")

files = [
    "Train/CL_train.json",
    "Train/CL_dev.json",
    "Train/IT_train.json",
    "Train/IT_dev.json",
    "Train/label_vocab.json",
    "Test/CL_test.json",
    "Test/IT_test.json",
]

for filename in files:
    hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        filename=f"Phase_3_RR/{filename}",
        local_dir=str(DATA_DIR),
        token=globals().get("HF_TOKEN") or os.getenv("HF_TOKEN"),
    )

RR_DIR = DATA_DIR / "Phase_3_RR"


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

In [11]:
checkpoint = torch.load(
    MODEL_DIR / "model.pt",
    map_location="cpu",
    weights_only=True,
)

cfg = WindowConfig(**checkpoint["model_config"])

phase1_model = WindowGQALM(cfg)

phase1_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

del checkpoint

MAX_LENGTH = min(MAX_LENGTH, cfg.max_seq_len)

state = load_json(MODEL_DIR / "tokenizer.json")

tokenizer = BytelevelBPE(state["vocab_size"])

tokenizer.specials = state["specials"]
tokenizer.vocab = state["vocab"]
tokenizer.vocab_size = len(tokenizer.vocab)

tokenizer.stoi = {
    symbol: i
    for i, symbol in enumerate(tokenizer.vocab)
}

tokenizer.itos = {
    i: symbol
    for symbol, i in tokenizer.stoi.items()
}

tokenizer.merges = [
    tuple(pair) for pair in state["merges"]
]

tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}

tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}

del state

PAD_ID = tokenizer.stoi["<pad>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]
UNK_ID = tokenizer.stoi["<unk>"]

if tokenizer.vocab_size != cfg.vocab_size:
    raise ValueError("Tokenizer vocabulary does not match the checkpoint.")

if PAD_ID != cfg.pad_id:
    raise ValueError("Tokenizer padding ID does not match the checkpoint.")

print("Input length:", MAX_LENGTH)
print("Attention window:", cfg.window_size)
print("Query heads:", cfg.n_heads)
print("Key/value heads:", cfg.n_kv_heads)

Input length: 512
Attention window: 256
Query heads: 8
Key/value heads: 2


In [12]:
vocab = load_json(RR_DIR / "Train/label_vocab.json")

if isinstance(vocab, dict):
    for key in ("label2id", "id2label", "labels"):
        if key in vocab and isinstance(vocab[key], (dict, list)):
            vocab = vocab[key]
            break

if isinstance(vocab, list):
    label_names = vocab
elif all(str(key).isdigit() for key in vocab):
    label_names = [
        vocab[key]
        for key in sorted(vocab, key=lambda x: int(x))
    ]
else:
    label_names = sorted(vocab, key=lambda x: int(vocab[x]))

label2id = {
    label: i for i, label in enumerate(label_names)
}

id2label = dict(enumerate(label_names))
NUM_LABELS = len(label_names)


def read_documents(path, domain, labeled=True):
    raw = load_json(path)

    if isinstance(raw, dict) and "id" in raw and "text" in raw:
        records = [raw]

    elif isinstance(raw, dict):
        records = []

        for doc_id, record in raw.items():
            if isinstance(record, list):
                record = {"text": record}

            records.append({**record, "id": doc_id})

    else:
        records = raw

    documents = []
    seen = set()

    for record in records:
        doc_id = str(record["id"])
        sentences = record["text"]
        labels = record["labels"] if labeled else None

        if doc_id in seen:
            raise ValueError(f"Duplicate document ID: {doc_id}")
        seen.add(doc_id)

        if (
            not isinstance(sentences, list)
            or any(not isinstance(s, str) for s in sentences)
        ):
            raise ValueError("Expected a list of sentence strings.")

        if labeled:
            if len(sentences) != len(labels):
                raise ValueError(f"Sentence-label mismatch: {doc_id}")

            if any(label not in label2id for label in labels):
                raise ValueError(f"Unknown label in document: {doc_id}")

        documents.append({
            "id": doc_id,
            "domain": domain,
            "text": sentences,
            "labels": labels,
        })

    return documents


train_docs = []
dev_docs = []

for domain in ("CL", "IT"):
    train_docs += read_documents(
        RR_DIR / f"Train/{domain}_train.json",
        domain,
    )

    dev_docs += read_documents(
        RR_DIR / f"Train/{domain}_dev.json",
        domain,
    )

print("Labels:", label_names)
print("Training documents:", len(train_docs))
print("Validation documents:", len(dev_docs))

Labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
Training documents: 80
Validation documents: 10


In [13]:
previous_prefix = tokenizer.encode("Previous sentence:")
next_prefix = tokenizer.encode("Next sentence:")
target_prefix = tokenizer.encode("Target sentence:")

overhead = (
    4
    + len(previous_prefix)
    + len(next_prefix)
    + len(target_prefix)
)

target_capacity = MAX_LENGTH - overhead

if target_capacity < 1:
    raise ValueError("MAX_LENGTH is too small for the input format.")


def make_chunks(target, left, right):
    chunks = []

    for start in range(0, len(target), target_capacity):
        part = target[start:start + target_capacity]
        available = target_capacity - len(part)

        n_left = min(len(left), available // 2)
        n_right = min(len(right), available - n_left)
        n_left = min(len(left), available - n_right)

        left_part = left[-n_left:] if n_left else []
        right_part = right[:n_right]

        ids = (
            [BOS_ID]
            + previous_prefix + left_part + [EOS_ID]
            + next_prefix + right_part + [EOS_ID]
            + target_prefix + part + [EOS_ID]
        )

        chunks.append((ids, len(part)))

    return chunks


class RRDataset(Dataset):
    def __init__(self, documents):
        self.documents = documents
        self.tokens = []
        self.index = []

        for doc_index, document in enumerate(documents):
            encoded = []

            for sentence_index, sentence in enumerate(document["text"]):
                ids = tokenizer.encode(sentence)
                encoded.append(ids or [UNK_ID])

                self.index.append(
                    (doc_index, sentence_index)
                )

            self.tokens.append(encoded)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, index):
        doc_index, sentence_index = self.index[index]

        document = self.documents[doc_index]
        sentences = self.tokens[doc_index]

        left = []
        right = []

        if USE_CONTEXT:
            if sentence_index > 0:
                left = sentences[sentence_index - 1]

            if sentence_index + 1 < len(sentences):
                right = sentences[sentence_index + 1]

        chunks = make_chunks(
            sentences[sentence_index],
            left,
            right,
        )

        label = -100

        if document["labels"] is not None:
            label = label2id[
                document["labels"][sentence_index]
            ]

        return {
            "chunks": chunks,
            "label": label,
            "position": sentence_index / max(1, len(sentences) - 1),
            "meta": (
                document["domain"],
                document["id"],
                sentence_index,
            ),
        }

In [14]:
def collate_rr(examples):
    input_ids = []
    lengths = []
    owners = []
    weights = []
    positions = []

    for owner, example in enumerate(examples):
        for ids, target_length in example["chunks"]:
            input_ids.append(
                torch.tensor(ids, dtype=torch.long)
            )

            lengths.append(len(ids))
            owners.append(owner)
            weights.append(target_length)
            positions.append(example["position"])

    return {
        "input_ids": pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=PAD_ID,
        ),
        "lengths": torch.tensor(lengths, dtype=torch.long),
        "owners": torch.tensor(owners, dtype=torch.long),
        "weights": torch.tensor(weights, dtype=torch.float32),
        "sentence_positions": torch.tensor(
            positions, dtype=torch.float32
        ),
        "labels": torch.tensor(
            [example["label"] for example in examples],
            dtype=torch.long,
        ),
        "meta": [example["meta"] for example in examples],
    }


train_dataset = RRDataset(train_docs)
dev_dataset = RRDataset(dev_docs)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_rr,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
)

print("Training sentences:", len(train_dataset))
print("Validation sentences:", len(dev_dataset))

Training sentences: 16845
Validation sentences: 2142


In [15]:
class LoRA(nn.Module):
    def __init__(self, base_layer, rank, alpha):
        super().__init__()

        self.base = base_layer
        self.base.requires_grad_(False)
        self.scale = alpha / rank

        self.A = nn.Linear(
            base_layer.in_features,
            rank,
            bias=False,
        ).to(base_layer.weight)

        self.B = nn.Linear(
            rank,
            base_layer.out_features,
            bias=False,
        ).to(base_layer.weight)

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return (
            self.base(x)
            + self.scale * self.B(self.A(x))
        )


class LoRMAPlus(nn.Module):
    def __init__(self, base_layer, rank, alpha):
        super().__init__()

        self.base = base_layer
        self.base.requires_grad_(False)
        self.scale = alpha / rank

        self.A = nn.Linear(
            base_layer.out_features,
            rank,
            bias=False,
        ).to(base_layer.weight)

        self.B = nn.Linear(
            rank,
            base_layer.out_features,
            bias=False,
        ).to(base_layer.weight)

        nn.init.normal_(self.A.weight, std=0.02)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        original = self.base(x)

        return (
            original
            + self.scale * self.B(self.A(original))
        )

In [16]:
class EfficientRR(nn.Module):
    def __init__(self, pretrained_model, num_labels, use_position):
        super().__init__()

        self.cfg = pretrained_model.cfg
        self.use_position = use_position

        self.embedding = pretrained_model.embedding
        self.embedding_dropout = pretrained_model.dropout
        self.blocks = pretrained_model.blocks
        self.final_norm = pretrained_model.final_norm

        self.dropout = nn.Dropout(self.cfg.dropout)

        feature_size = (
            2 * self.cfg.d_model + int(use_position)
        )

        self.classifier = nn.Linear(
            feature_size,
            num_labels,
        )

    def encode_sentences(
        self,
        input_ids,
        lengths,
        target_lengths,
        sentence_positions,
    ):
        positions = torch.arange(
            input_ids.size(1),
            device=input_ids.device,
        )

        valid = input_ids.ne(self.cfg.pad_id)

        x = self.embedding_dropout(
            self.embedding(input_ids)
        )

        for block in self.blocks:
            x, _ = block(
                x,
                positions,
                valid,
                past=None,
                use_cache=False,
            )

        x = self.final_norm(x).float()

        rows = torch.arange(x.size(0), device=x.device)

        eos_positions = lengths - 1
        eos_vector = x[rows, eos_positions]

        target_starts = eos_positions - target_lengths

        target_mask = (
            (positions[None, :] >= target_starts[:, None])
            & (positions[None, :] < eos_positions[:, None])
            & valid
        )

        mask = target_mask.unsqueeze(-1).float()

        target_mean = (
            (x * mask).sum(dim=1)
            / mask.sum(dim=1).clamp_min(1)
        )

        features = torch.cat(
            [target_mean, eos_vector],
            dim=-1,
        )

        if self.use_position:
            features = torch.cat(
                [
                    features,
                    sentence_positions.float().unsqueeze(-1),
                ],
                dim=-1,
            )

        return features

    def forward(
        self,
        input_ids,
        lengths,
        target_lengths,
        sentence_positions,
    ):
        features = self.encode_sentences(
            input_ids,
            lengths,
            target_lengths,
            sentence_positions,
        )

        # Apply dropout to text features; retain the position scalar
        if self.use_position:
            features = torch.cat(
                [
                    self.dropout(features[:, :-1]),
                    features[:, -1:],
                ],
                dim=-1,
            )
        else:
            features = self.dropout(features)

        return self.classifier(features)

In [17]:
torch.manual_seed(SEED)

rr_model = EfficientRR(
    phase1_model,
    NUM_LABELS,
    use_position=USE_POSITION,
)

del phase1_model

rr_model.requires_grad_(False)

Adapter = {
    "lora": LoRA,
    "lorma_plus": LoRMAPlus,
}[METHOD]

for block in rr_model.blocks:
    for name in ATTENTION_TARGETS:
        layer = getattr(block.attention, name)

        setattr(
            block.attention,
            name,
            Adapter(layer, rank=RANK, alpha=ALPHA),
        )

    for name in FFN_TARGETS:
        layer = getattr(block.ffn, name)

        setattr(
            block.ffn,
            name,
            Adapter(layer, rank=RANK, alpha=ALPHA),
        )

rr_model.classifier.requires_grad_(True)
rr_model = rr_model.to(device)

total = sum(p.numel() for p in rr_model.parameters())

trainable = sum(
    p.numel()
    for p in rr_model.parameters()
    if p.requires_grad
)

print("Method:", METHOD)
print("Attention targets:", ATTENTION_TARGETS)
print("FFN targets:", FFN_TARGETS)
print("Classifier input:", rr_model.classifier.in_features)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Trainable percentage: {100 * trainable / total:.2f}%")

Method: lora
Attention targets: ['q_proj', 'k_proj', 'v_proj', 'out_proj']
FFN targets: ['gate', 'up', 'down']
Classifier input: 1024
Total parameters: 51,815,949
Trainable parameters: 1,070,093
Trainable percentage: 2.07%


In [18]:
def move_batch(batch):
    return {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def get_sentence_logits(model, batch):
    chunk_logits = model(
        batch["input_ids"],
        batch["lengths"],
        batch["weights"].long(),
        batch["sentence_positions"],
    ).float()

    owners = batch["owners"]
    weights = batch["weights"]

    logits = chunk_logits.new_zeros(
        len(batch["meta"]),
        NUM_LABELS,
    )

    logits = logits.index_add(
        0,
        owners,
        chunk_logits * weights[:, None],
    )

    totals = weights.new_zeros(len(batch["meta"]))
    totals = totals.index_add(0, owners, weights)

    return logits / totals[:, None]


counts = torch.zeros(NUM_LABELS)

for document in train_docs:
    for label in document["labels"]:
        counts[label2id[label]] += 1

class_weights = None

if USE_CLASS_WEIGHTS:
    class_weights = counts.clamp_min(1).pow(-0.25)
    class_weights /= class_weights.mean()
    class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

print(dict(zip(label_names, counts.int().tolist())))

{'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'RulingByPresentCourt': 307, 'None': 297}


In [19]:
def macro_scores(gold, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        gold,
        predicted,
        labels=list(range(NUM_LABELS)),
        average="macro",
        zero_division=0,
    )

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


@torch.inference_mode()
def evaluate_rr(model, loader):
    model.eval()

    gold = []
    predicted = []
    domains = []

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_sentence_logits(model, batch)

        gold.extend(batch["labels"].tolist())
        predicted.extend(logits.argmax(dim=-1).tolist())
        domains.extend(meta[0] for meta in batch["meta"])

    scores = macro_scores(gold, predicted)
    scores["domains"] = {}

    for domain in ("CL", "IT"):
        indices = [
            i for i, name in enumerate(domains)
            if name == domain
        ]

        if indices:
            scores["domains"][domain] = macro_scores(
                [gold[i] for i in indices],
                [predicted[i] for i in indices],
            )

    return scores, gold, predicted

In [20]:
trainable_parameters = [
    p for p in rr_model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp and amp_dtype == torch.float16,
)

run_config = {
    "method": METHOD,
    "adapter_class": Adapter.__name__,
    "rank": RANK,
    "alpha": ALPHA,
    "attention_targets": ATTENTION_TARGETS,
    "ffn_targets": FFN_TARGETS,
    "max_length": MAX_LENGTH,
    "window_size": cfg.window_size,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "use_context": USE_CONTEXT,
    "use_position": USE_POSITION,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "class_weight_power": 0.25,
    "pooling": "target_mean_plus_eos",
    "input_order": "previous_next_target",
    "seed": SEED,
    "backbone": "WindowGQALM",
    "causal": True,
}

(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2),
    encoding="utf-8",
)

history = []
best_f1 = -1.0
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    rr_model.train()

    total_loss = 0.0
    total_weight = 0.0

    for step, batch in enumerate(train_loader, 1):
        batch = move_batch(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_sentence_logits(rr_model, batch)
            loss = criterion(logits, batch["labels"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_parameters, 1.0
        )

        scaler.step(optimizer)
        scaler.update()

        weight = (
            len(batch["meta"])
            if class_weights is None
            else class_weights[batch["labels"]].sum().item()
        )

        total_loss += loss.item() * weight
        total_weight += weight

        if step % 500 == 0:
            print(
                f"Epoch {epoch} | "
                f"Batch {step}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    scores, _, _ = evaluate_rr(rr_model, dev_loader)
    train_loss = total_loss / total_weight

    print(
        f"Epoch {epoch} | "
        f"Train loss: {train_loss:.4f} | "
        f"Dev Macro-F1: {scores['f1']:.4f}"
    )
    print("Domain scores:", scores["domains"])

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **scores,
    })

    (OUTPUT_DIR / "history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8",
    )

    if scores["f1"] > best_f1:
        best_f1 = scores["f1"]
        bad_epochs = 0

        torch.save(
            {
                "model_state_dict": {
                    name: tensor.detach().cpu()
                    for name, tensor in rr_model.state_dict().items()
                },
                "model_config": asdict(cfg),
                "run_config": run_config,
                "label2id": label2id,
                "epoch": epoch,
                "dev_scores": scores,
            },
            OUTPUT_DIR / "best_f1.pt",
        )

        print("Saved best checkpoint.")

    else:
        bad_epochs += 1

        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

Epoch 1 | Batch 500/1053 | Loss: 1.9805
Epoch 1 | Batch 1000/1053 | Loss: 0.8631
Epoch 1 | Train loss: 1.4695 | Dev Macro-F1: 0.3259
Domain scores: {'CL': {'precision': 0.3643582885621305, 'recall': 0.28507757058942473, 'f1': 0.2731747262795689}, 'IT': {'precision': 0.4815111724949741, 'recall': 0.3360883515376988, 'f1': 0.34124857213833537}}
Saved best checkpoint.
Epoch 2 | Batch 500/1053 | Loss: 0.7203
Epoch 2 | Batch 1000/1053 | Loss: 0.8984
Epoch 2 | Train loss: 1.1097 | Dev Macro-F1: 0.3842
Domain scores: {'CL': {'precision': 0.3108977220821326, 'recall': 0.2819524097237499, 'f1': 0.27576590126428085}, 'IT': {'precision': 0.5023555173413438, 'recall': 0.3892916398103041, 'f1': 0.4186092209823592}}
Saved best checkpoint.
Epoch 3 | Batch 500/1053 | Loss: 0.5197
Epoch 3 | Batch 1000/1053 | Loss: 0.9008
Epoch 3 | Train loss: 0.9351 | Dev Macro-F1: 0.3669
Domain scores: {'CL': {'precision': 0.28924054021069406, 'recall': 0.28039990217529814, 'f1': 0.2708471712856809}, 'IT': {'precision

In [21]:
checkpoint = torch.load(
    OUTPUT_DIR / "best_f1.pt",
    map_location="cpu",
    weights_only=True,
)

rr_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

print("Best epoch:", checkpoint["epoch"])
del checkpoint

scores, gold, predicted = evaluate_rr(
    rr_model, dev_loader
)

print(
    classification_report(
        gold,
        predicted,
        labels=list(range(NUM_LABELS)),
        target_names=label_names,
        digits=4,
        zero_division=0,
    )
)

print("Validation scores:", scores)

Best epoch: 2
                        precision    recall  f1-score   support

                  Fact     0.5852    0.6419    0.6122       444
                 Issue     0.7778    0.4667    0.5833        15
    ArgumentPetitioner     0.6604    0.2625    0.3757       400
    ArgumentRespondent     0.4781    0.4007    0.4360       272
               Statute     0.9762    0.6833    0.8039       120
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.4709    0.3504    0.4018       254
PrecedentNotReliedUpon     0.3333    0.2600    0.2921        50
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.4634    0.1863    0.2657       102
    RatioOfTheDecision     0.3745    0.7742    0.5049       403
  RulingByPresentCourt     0.4600    0.4694    0.4646        49
                  None     0.3182    0.2121    0.2545        33

              accuracy                         0.4907      2142
             macro avg  

In [22]:
test_docs = []

for domain in ("CL", "IT"):
    test_docs += read_documents(
        RR_DIR / f"Test/{domain}_test.json",
        domain,
        labeled=False,
    )

test_dataset = RRDataset(test_docs)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_rr,
)


@torch.inference_mode()
def predict_rr(model, loader, documents):
    model.eval()

    predictions = {
        (doc["domain"], doc["id"]): [None] * len(doc["text"])
        for doc in documents
    }

    for batch in loader:
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            logits = get_sentence_logits(model, batch)

        predicted_ids = logits.argmax(dim=-1).tolist()

        for meta, label_id in zip(batch["meta"], predicted_ids):
            domain, doc_id, sentence_index = meta

            predictions[(domain, doc_id)][sentence_index] = (
                id2label[label_id]
            )

    return predictions


predictions = predict_rr(
    rr_model,
    test_loader,
    test_docs,
)

submission = {}

for (domain, doc_id), labels in predictions.items():
    if doc_id in submission:
        raise ValueError(f"Duplicate document ID: {doc_id}")

    if any(label is None for label in labels):
        raise ValueError(f"Missing predictions for document: {doc_id}")

    submission[doc_id] = labels

prediction_path = OUTPUT_DIR / "prediction.json"

prediction_path.write_text(
    json.dumps(submission, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

with zipfile.ZipFile(
    OUTPUT_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname="prediction.json",
    )

print("Saved:", OUTPUT_DIR / "submission.zip")

Saved: /kaggle/working/phase3/efficient_lora_mean_eos_position0/submission.zip


# **Bigru**

In [23]:
SENTENCE_CHECKPOINT = OUTPUT_DIR / "best_f1.pt"

checkpoint = torch.load(
    SENTENCE_CHECKPOINT,
    map_location="cpu",
    weights_only=True,
)

rr_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

print("Best sentence-model epoch:", checkpoint["epoch"])
print("Sentence-model scores:", checkpoint["dev_scores"])

del checkpoint

rr_model = rr_model.to(device)
rr_model.requires_grad_(False)
rr_model.eval()

BIGRU_DIR = OUTPUT_DIR / "document_bigru"
BIGRU_DIR.mkdir(parents=True, exist_ok=True)

DOC_BATCH_SIZE = 4
GRU_HIDDEN = 512
GRU_DROPOUT = 0.2
GRU_LR = 1e-3
GRU_EPOCHS = 15
GRU_PATIENCE = 3

Best sentence-model epoch: 2
Sentence-model scores: {'precision': 0.4536931130873722, 'recall': 0.362115224016247, 'f1': 0.38422245346926753, 'domains': {'CL': {'precision': 0.3108977220821326, 'recall': 0.2819524097237499, 'f1': 0.27576590126428085}, 'IT': {'precision': 0.5023555173413438, 'recall': 0.3892916398103041, 'f1': 0.4186092209823592}}}


In [24]:
@torch.no_grad()
def extract_document_features(model, documents):
    model.eval()

    loader = DataLoader(
        RRDataset(documents),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_rr,
    )

    feature_size = model.classifier.in_features

    document_features = {
        (doc["domain"], doc["id"]): torch.zeros(
            len(doc["text"]),
            feature_size,
        )
        for doc in documents
    }

    for step, batch in enumerate(loader, 1):
        batch = move_batch(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=use_amp,
        ):
            chunk_features = model.encode_sentences(
                batch["input_ids"],
                batch["lengths"],
                batch["weights"].long(),
                batch["sentence_positions"],
            ).float()

        owners = batch["owners"]
        weights = batch["weights"]

        features = chunk_features.new_zeros(
            len(batch["meta"]),
            feature_size,
        )

        features.index_add_(
            0,
            owners,
            chunk_features * weights[:, None],
        )

        totals = weights.new_zeros(len(batch["meta"]))
        totals.index_add_(0, owners, weights)

        features = (features / totals[:, None]).cpu()

        for meta, vector in zip(batch["meta"], features):
            domain, doc_id, sentence_index = meta

            document_features[
                (domain, doc_id)
            ][sentence_index] = vector

        if step % 100 == 0:
            print(f"Extracted {step}/{len(loader)} batches")

    records = []

    for doc in documents:
        if not doc["text"]:
            continue

        key = (doc["domain"], doc["id"])

        labels = (
            [label2id[label] for label in doc["labels"]]
            if doc["labels"] is not None
            else [-100] * len(doc["text"])
        )

        records.append({
            "features": document_features[key],
            "labels": torch.tensor(labels, dtype=torch.long),
            "key": key,
        })

    return records

train_records = extract_document_features(
    rr_model, train_docs
)

dev_records = extract_document_features(
    rr_model, dev_docs
)

Extracted 100/1053 batches
Extracted 200/1053 batches
Extracted 300/1053 batches
Extracted 400/1053 batches
Extracted 500/1053 batches
Extracted 600/1053 batches
Extracted 700/1053 batches
Extracted 800/1053 batches
Extracted 900/1053 batches
Extracted 1000/1053 batches
Extracted 100/134 batches


In [25]:
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
)


def collate_documents(records):
    return {
        "features": pad_sequence(
            [record["features"] for record in records],
            batch_first=True,
            padding_value=0.0,
        ),
        "labels": pad_sequence(
            [record["labels"] for record in records],
            batch_first=True,
            padding_value=-100,
        ),
        "lengths": torch.tensor(
            [len(record["labels"]) for record in records],
            dtype=torch.long,
        ),
        "keys": [record["key"] for record in records],
    }


doc_train_loader = DataLoader(
    train_records,
    batch_size=DOC_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_documents,
)

doc_dev_loader = DataLoader(
    dev_records,
    batch_size=DOC_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_documents,
)


class DocumentBiGRU(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_labels,
        dropout=0.2,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            2 * hidden_size,
            num_labels,
        )

    def forward(self, features, lengths):
        packed = pack_padded_sequence(
            features,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        packed_output, _ = self.gru(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=features.size(1),
        )

        return self.classifier(self.dropout(output))


torch.manual_seed(SEED)

doc_model = DocumentBiGRU(
    input_size=rr_model.classifier.in_features,
    hidden_size=GRU_HIDDEN,
    num_labels=NUM_LABELS,
    dropout=GRU_DROPOUT,
).to(device)

In [26]:
@torch.no_grad()
def evaluate_bigru(model, loader):
    model.eval()

    gold = []
    predicted = []

    for batch in loader:
        features = batch["features"].to(device)
        labels = batch["labels"].to(device)

        logits = model(features, batch["lengths"])
        predictions = logits.argmax(dim=-1)

        valid = labels.ne(-100)

        gold.extend(labels[valid].cpu().tolist())
        predicted.extend(predictions[valid].cpu().tolist())

    return macro_scores(gold, predicted), gold, predicted


doc_criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=-100,
)

doc_optimizer = torch.optim.AdamW(
    doc_model.parameters(),
    lr=GRU_LR,
    weight_decay=0.01,
)

best_doc_f1 = -1.0
bad_epochs = 0

for epoch in range(1, GRU_EPOCHS + 1):
    doc_model.train()

    total_loss = 0.0
    total_weight = 0.0

    for batch in doc_train_loader:
        features = batch["features"].to(device)
        labels = batch["labels"].to(device)

        doc_optimizer.zero_grad(set_to_none=True)

        logits = doc_model(features, batch["lengths"])

        loss = doc_criterion(
            logits.reshape(-1, NUM_LABELS),
            labels.reshape(-1),
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            doc_model.parameters(), 1.0
        )

        doc_optimizer.step()

        valid_labels = labels[labels.ne(-100)]

        weight = (
            valid_labels.numel()
            if class_weights is None
            else class_weights[valid_labels].sum().item()
        )

        total_loss += loss.item() * weight
        total_weight += weight

    scores, _, _ = evaluate_bigru(
        doc_model, doc_dev_loader
    )

    print(
        f"Epoch {epoch} | "
        f"Train loss: {total_loss / total_weight:.4f} | "
        f"Dev Macro-F1: {scores['f1']:.4f}"
    )

    if scores["f1"] > best_doc_f1:
        best_doc_f1 = scores["f1"]
        bad_epochs = 0

        torch.save(
            {
                "model_state_dict": {
                    name: tensor.detach().cpu()
                    for name, tensor in doc_model.state_dict().items()
                },
                "input_size": rr_model.classifier.in_features,
                "hidden_size": GRU_HIDDEN,
                "dropout": GRU_DROPOUT,
                "num_labels": NUM_LABELS,
                "label2id": label2id,
                "pooling": "target_mean_plus_eos",
                "sentence_checkpoint": str(SENTENCE_CHECKPOINT),
                "epoch": epoch,
                "dev_scores": scores,
                "use_position": rr_model.use_position,
            },
            BIGRU_DIR / "best_bigru.pt",
        )

        print("Saved best BiGRU checkpoint.")

    else:
        bad_epochs += 1

        if bad_epochs >= GRU_PATIENCE:
            print("Early stopping.")
            break

Epoch 1 | Train loss: 1.3594 | Dev Macro-F1: 0.3905
Saved best BiGRU checkpoint.
Epoch 2 | Train loss: 0.8483 | Dev Macro-F1: 0.3584
Epoch 3 | Train loss: 0.7234 | Dev Macro-F1: 0.4356
Saved best BiGRU checkpoint.
Epoch 4 | Train loss: 0.6285 | Dev Macro-F1: 0.3860
Epoch 5 | Train loss: 0.5465 | Dev Macro-F1: 0.4108
Epoch 6 | Train loss: 0.4386 | Dev Macro-F1: 0.4365
Saved best BiGRU checkpoint.
Epoch 7 | Train loss: 0.3885 | Dev Macro-F1: 0.4511
Saved best BiGRU checkpoint.
Epoch 8 | Train loss: 0.3614 | Dev Macro-F1: 0.4479
Epoch 9 | Train loss: 0.2977 | Dev Macro-F1: 0.4548
Saved best BiGRU checkpoint.
Epoch 10 | Train loss: 0.2589 | Dev Macro-F1: 0.4645
Saved best BiGRU checkpoint.
Epoch 11 | Train loss: 0.2066 | Dev Macro-F1: 0.4618
Epoch 12 | Train loss: 0.1819 | Dev Macro-F1: 0.4254
Epoch 13 | Train loss: 0.1766 | Dev Macro-F1: 0.4610
Early stopping.


In [27]:
checkpoint = torch.load(
    BIGRU_DIR / "best_bigru.pt",
    map_location="cpu",
    weights_only=True,
)

doc_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

print("Best BiGRU epoch:", checkpoint["epoch"])
del checkpoint

scores, gold, predicted = evaluate_bigru(
    doc_model, doc_dev_loader
)

print("BiGRU validation macro F1:", scores["f1"])

print(
    classification_report(
        gold,
        predicted,
        labels=list(range(NUM_LABELS)),
        target_names=label_names,
        digits=4,
        zero_division=0,
    )
)

Best BiGRU epoch: 10
BiGRU validation macro F1: 0.4644787918759157
                        precision    recall  f1-score   support

                  Fact     0.7074    0.7568    0.7312       444
                 Issue     0.6923    0.6000    0.6429        15
    ArgumentPetitioner     0.7239    0.6750    0.6986       400
    ArgumentRespondent     0.7880    0.5331    0.6360       272
               Statute     0.8000    0.7333    0.7652       120
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.5199    0.5669    0.5424       254
PrecedentNotReliedUpon     0.1408    0.2000    0.1653        50
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.8611    0.3039    0.4493       102
    RatioOfTheDecision     0.6059    0.8089    0.6929       403
  RulingByPresentCourt     0.5385    0.4286    0.4773        49
                  None     0.2692    0.2121    0.2373        33

              accuracy             

In [28]:
test_docs = []

for domain in ("CL", "IT"):
    test_docs += read_documents(
        RR_DIR / f"Test/{domain}_test.json",
        domain,
        labeled=False,
    )

test_records = extract_document_features(
    rr_model, test_docs
)

doc_test_loader = DataLoader(
    test_records,
    batch_size=DOC_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_documents,
)


@torch.no_grad()
def predict_bigru(model, loader, documents):
    model.eval()

    predictions = {
        (doc["domain"], doc["id"]): []
        for doc in documents
    }

    for batch in loader:
        features = batch["features"].to(device)

        logits = model(features, batch["lengths"])
        predicted_ids = logits.argmax(dim=-1).cpu()

        for row, (key, length) in enumerate(
            zip(batch["keys"], batch["lengths"].tolist())
        ):
            predictions[key] = [
                id2label[label_id]
                for label_id in predicted_ids[row, :length].tolist()
            ]

    return predictions


predictions = predict_bigru(
    doc_model,
    doc_test_loader,
    test_docs,
)

submission = {}

for (domain, doc_id), labels in predictions.items():
    if doc_id in submission:
        raise ValueError(f"Duplicate document ID: {doc_id}")

    submission[doc_id] = labels

prediction_path = BIGRU_DIR / "prediction.json"

prediction_path.write_text(
    json.dumps(submission, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

with zipfile.ZipFile(
    BIGRU_DIR / "submission.zip",
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        prediction_path,
        arcname="prediction.json",
    )

print("Saved:", BIGRU_DIR / "submission.zip")

Extracted 100/138 batches
Saved: /kaggle/working/phase3/efficient_lora_mean_eos_position0/document_bigru/submission.zip
